# Study v2, Step 1: prompts, thinking budgets, seeds, plausible swaps (Colab runner)

**Runtime → Change runtime type → A100 GPU.** Plan: `STUDY_V2.md`. Rough GPU time on an A100 40 GB (estimated from the
v1 runs, so treat it as a guess): setup and smoke test 15 min, plausible swaps 10 min, **each model about 1.5 h for the
core parts and 45 min for the extra parts**, the trace judge about 30 min.

Suggested sessions (each can run while you work on something else; keep this tab open):
1. Sections 1–3 (setup, tests, download helper), then section 4 (plausible swaps) and section 5 (Qwen3-8B).
2. New session: sections 1 and 3, then section 6 (Qwen3-14B).
3. Later (optional, after the hand labels): sections 1 and 3, then section 7 (judge).
4. Short session to finish Step 1: section 1, then section 7b.

Everything is resumable: after a disconnect, run sections 1 and 3 again, then the cell that stopped. Finished
questions are skipped. **Section 3 defines `save()`**, which downloads a zip of `results_v2/` to your computer; the
run cells call it after every part, so a disconnect loses at most one part. Your browser may ask once whether this
site may download several files: allow it. Before a new session, upload the zips of the earlier sessions to the file
panel; the last cell of section 1 restores them. When you are done: **Runtime → Disconnect and delete runtime**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 1. Get the code
Either upload `rag-knowledge-conflict-v2-step1.zip` to the Colab file panel (left side) before running this cell, or,
once the `v2-step1` branch is on GitHub, the cell clones it.

In [ ]:
import os
os.chdir('/content')
if os.path.exists('/content/rag-knowledge-conflict-v2-step1.zip'):
    !rm -rf /content/rag-knowledge-conflict && mkdir -p /content/rag-knowledge-conflict
    !unzip -q -o /content/rag-knowledge-conflict-v2-step1.zip -d /content/rag-knowledge-conflict
elif os.path.exists('/content/rag-knowledge-conflict/.git'):
    !git -C /content/rag-knowledge-conflict pull -q   # already cloned: get the latest version
else:
    !git clone -q -b v2-step1 https://github.com/SoroushJaberi/rag-knowledge-conflict.git /content/rag-knowledge-conflict
%cd /content/rag-knowledge-conflict
!ls

In [ ]:
# Installs vLLM and the data tools (a few minutes). If Colab asks you to restart the session, restart, then run
# the "Get the code" cell again and continue with the next cell after this one.
!pip install -q vllm datasets nvidia-ml-py

In [ ]:
# Rebuild the 1,000-question evaluation set and check it is the published one
!python prepare_data.py > /dev/null
!sha256sum data/items.jsonl
print("expected: 0ad33638bebc7e92598fa0ec0829eb0f0527204fb17c096f883e48fb4f76683f")

In [ ]:
# Restore results from earlier sessions: upload results_v2.zip files to the file panel first (skip in session 1)
import glob
for z in sorted(glob.glob('/content/results_v2*.zip')):
    print('restoring', z)
    !unzip -q -o "{z}" -d /content/rag-knowledge-conflict
!ls results_v2 2>/dev/null | head -50

### 2. Tests (no GPU), then a 5-minute smoke test on Qwen3-0.6B that exercises budgets and "Wait"

In [ ]:
!python test_scoring.py && python test_v2.py && python test_pipeline.py --data data/items.jsonl --work /tmp/pipeline_test > /tmp/pipeline_test.log && echo "pipeline test OK"
!rm -rf smoke_v2
!python run_plan.py --model Qwen/Qwen3-0.6B --parts base,budgets,wait --limit 5 --out smoke_v2
import json
for run in ['Qwen3-0.6B_think_b0_s14', 'Qwen3-0.6B_think_b128_s14', 'Qwen3-0.6B_think_wait1_s14']:
    p = json.loads(open(f'smoke_v2/{run}/predictions.jsonl').readline())
    print(f"\n{run}: answer={p['answer']!r} thinking tokens={p['n_thinking']} stopped by budget={p['forced_stop']}")
    print('   ...', p['reasoning'][-200:].replace('\n', ' '))

### 3. Download helper
`save()` zips `results_v2/` and the plausible-swap files and downloads them. Keep every downloaded zip.

In [ ]:
from google.colab import files
def save(name='results_v2.zip'):
    !rm -f {name}
    !zip -r -q {name} results_v2 data/plausible_answers.jsonl data/plausible_meta.json data/plausible_checks.jsonl 2>/dev/null
    files.download(name)

### 4. Plausible wrong answers (about 10 min; once)
Generator: `microsoft/phi-4` (MIT licence, not a Qwen model). Writes `data/plausible_answers.jsonl` (stored in the
repository afterwards, so later runs do not need the generator).

In [ ]:
!test -f data/plausible_answers.jsonl || python plausible.py generate   # never regenerates existing answers
!python plausible.py check
save('results_v2_plausible.zip')

### 5. Qwen3-8B (session 1)
Core parts: base runs (seed 14), prompt variants, thinking budgets. Extra parts: second seed (15), plausible swaps,
"Wait" extension.

In [ ]:
!python run_plan.py --model Qwen/Qwen3-8B --parts base,prompts,budgets
save('results_v2_8B_core.zip')
!python run_plan.py --model Qwen/Qwen3-8B --parts seeds,plausible,wait
save('results_v2_8B_all.zip')

### 6. Qwen3-14B (session 2)
The first line checks the plausible swaps once (phi-4, about 4 min): questions whose proposed wrong answer may also be correct are removed.

In [ ]:
!test -f data/plausible_checks.jsonl || python plausible.py verify
!python run_plan.py --model Qwen/Qwen3-14B --parts base,prompts,budgets
save('results_v2_14B_core.zip')
!python run_plan.py --model Qwen/Qwen3-14B --parts seeds,plausible,wait
save('results_v2_14B_all.zip')

### 7. Trace judge (session 3, optional; about 30 min)
Needs all runs: upload the zips from sessions 1 and 2 before running section 1. The judge (`microsoft/phi-4`) answers the
three questions of the hand-label sheet for up to 300 known-fact traces per condition and run, and for every trace in
`labels/hand_labels.csv`.

In [ ]:
!python trace_labels.py judge
save('results_v2_judge.zip')

### 7b. Finish Step 1 (short session, about 15 min)
In a fresh session, after section 1 (no uploads needed): checks the plausible swaps once (phi-4), and runs the judge on the two v1 runs whose traces are in the hand-label sheet (skipped if their labels are already in the repository). Downloads `results_v2_finish.zip`.

In [ ]:
from google.colab import files
!test -f data/plausible_checks.jsonl || python plausible.py verify
!python trace_labels.py judge --runs results
!zip -q -r results_v2_finish.zip data/plausible_checks.jsonl results/Qwen3-8B_think/judge*.jsonl results/Qwen3-14B_think/judge*.jsonl
files.download('results_v2_finish.zip')

### 8. Analysis
Runs anywhere without a GPU (also on your laptop): `python analyze_v2.py` → `results_v2/summary_step1.md`.

In [ ]:
!pip install -q matplotlib
!python analyze_v2.py > /dev/null
from IPython.display import Markdown, display
display(Markdown(open('results_v2/summary_step1.md', encoding='utf-8').read()))